In [1]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [3]:
pd.__version__

'2.3.3'

In [2]:
df = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet')
# Question 1: Number of columns
print(f"Number of columns: {len(df.columns)}")

Number of columns: 19


In [9]:
!pip install pyarrow


In [12]:
df.head()



,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [9]:
import sklearn

In [10]:
sklearn.__version__

'1.0.2'

NameError: name 'df' is not defined

In [7]:
# Q2 and Q3
df_train = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet')
df_val = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet')
original_train_len = len(df_train)

for df in (df_train, df_val):
    df['duration'] = (pd.to_datetime(df.tpep_dropoff_datetime) - pd.to_datetime(df.tpep_pickup_datetime)).dt.total_seconds() / 60

print(f'Q2 std: {df_train.duration.std():.2f}')

df_train = df_train[(df_train.duration >= 1) & (df_train.duration <= 60)].copy()
df_val = df_val[(df_val.duration >= 1) & (df_val.duration <= 60)].copy()
print(f'Q3 fraction left: {len(df_train) / original_train_len:.4f}')

Q2 std: 42.59
Q3 fraction left: 0.9812


In [8]:
# Q4
categorical = ['PULocationID', 'DOLocationID']
train_q4 = df_train[categorical].copy()
val_q4 = df_val[categorical].copy()

for df in (train_q4, val_q4):
    df[categorical] = df[categorical].astype(str)

dv = DictVectorizer()
X_train = dv.fit_transform(train_q4.to_dict(orient='records'))
X_val = dv.transform(val_q4.to_dict(orient='records'))

print(f'Q4 dimensionality: {X_train.shape[1]}')

Q4 dimensionality: 515


In [ ]:
# Q5
y_train = df_train['duration'].values
lr = LinearRegression()
lr.fit(X_train, y_train)
train_pred = lr.predict(X_train)

print(f'Q5 RMSE train: {root_mean_squared_error(y_train, train_pred):.2f}')

In [6]:
# Q6
y_val = df_val['duration'].values
val_pred = lr.predict(X_val)

print(f'Q6 RMSE validation: {root_mean_squared_error(y_val, val_pred):.2f}')

Q6 RMSE validation: 42.28
